# 6. The FMM operator chain

`UniformFmm` composes six operators. This tutorial builds each one with the
public Cartesian operator functions, checks it against the exact direct sum,
and finishes by inspecting the multipole and local coefficients of a real
static plan. The conventions are those of the mathematics pages: Green
function $G(r) = 1/(4\pi|r|)$, field $H = -\nabla\phi$, and displacements as
in the table.

| Operator | Function | Input → output | Displacement |
|---|---|---|---|
| P2P | `p2p_dipole_pair`, `p2p_dipole_sum` | moments → exact $\phi$, $H$ at a target | $r = x_{\rm target} - x_{\rm source}$ |
| P2M | `p2m_dipole` | moments in a box → multipole $M$ about the box centre $c_s$ | $d_j = x_j - c_s$ |
| M2M | `m2m` | child multipole → parent multipole | $d = c_{\rm parent} - c_{\rm child}$ |
| M2P | `m2p` | multipole → $\phi$, $H$ at a distant target (validation only) | $R = x - c_s$ |
| M2L | `m2l` | multipole → local expansion $L$ about a target centre $c_t$ | $R = c_t - c_s$ |
| L2L | `l2l` | parent local → child local | $d = c_{\rm child} - c_{\rm parent}$ |
| L2P | `l2p` | local expansion → $\phi$, $H$ at a target in the box | $dx = x - c_t$ |

The Python operator functions use the Cartesian Taylor basis with
factorial-normalised monomials; `multi_indices(p)` gives the coefficient
ordering and `spherical_modes(p)` the $(l, m)$ ordering of the real spherical
basis that the static plans use by default.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import cdfmm
from tutorial_utils import (
    direct_fields, draw_box_3d, error_metrics, finish_3d_axes, local_fields,
    multipole_fields, new_3d_figure, plot_coefficients_by_degree,
    quiet_construction, random_unit_vectors, relative_error, vec3_to_array,
)

plt.rcParams.update({"figure.dpi": 100, "axes.grid": True})
rng = np.random.default_rng(42)

## P2P: the exact pair interaction

For a unit moment along $x$, the axial field at unit distance is
$2/(4\pi)$ along the moment and the transverse field is $-1/(4\pi)$.

In [ ]:
source = np.zeros(3)
moment = np.array([1.0, 0.0, 0.0])
constant = 1.0 / (4.0 * np.pi)
axial = cdfmm.p2p_dipole_pair([1.0, 0.0, 0.0], source, moment, output="both")
transverse = cdfmm.p2p_dipole_pair([0.0, 1.0, 0.0], source, moment, output="both")
print(f"axial:      phi = {axial['phi']:.6f} (expected {constant:.6f}), H = {axial['H']}")
print(f"transverse: phi = {transverse['phi']:.6f} (expected 0),        H = {transverse['H']}")

coordinates = np.linspace(-2.0, 2.0, 23)
X, Y = np.meshgrid(coordinates, coordinates)
field = np.full(X.shape + (3,), np.nan)
for row in range(X.shape[0]):
    for column in range(X.shape[1]):
        point = np.array([X[row, column], Y[row, column], 0.0])
        if np.linalg.norm(point) < 0.12:
            continue                     # the singular source point
        field[row, column] = cdfmm.p2p_dipole_pair(point, source, moment, output="field")["H"]
magnitude = np.linalg.norm(field[..., :2], axis=-1)
scale = np.minimum(1.0, np.nanpercentile(magnitude, 85) / magnitude)
figure, axes = plt.subplots(figsize=(6.5, 5.5))
quiver = axes.quiver(X, Y, field[..., 0] * scale, field[..., 1] * scale, np.log10(magnitude),
                     cmap="viridis", pivot="mid")
axes.arrow(0.0, 0.0, 0.35, 0.0, width=0.025, color="tab:red")
axes.set(aspect="equal", xlabel="x", ylabel="y", title="Field of one dipole in the z = 0 plane")
figure.colorbar(quiver, ax=axes, label=r"$\log_{10}|H_{xy}|$")
figure.tight_layout()

## P2M: compress a source box into multipole coefficients

Dipoles contribute through $\alpha - e_k$, so the monopole coefficient
$M_{(0,0,0)}$ is exactly zero. The Cartesian basis of order $p$ has
$(p+1)(p+2)(p+3)/6$ coefficients ordered by total degree; the real spherical
basis has $(p+1)^2$.

In [ ]:
order = 4
source_centre = np.zeros(3)
source_positions = rng.uniform(-0.4, 0.4, size=(60, 3))
dipole_moments = rng.normal(size=(60, 3))

multipole = cdfmm.p2m_dipole(source_centre, source_positions, dipole_moments, order=order)
indices = cdfmm.multi_indices(order)
print(f"order {order}: {len(multipole)} Cartesian coefficients, "
      f"{len(cdfmm.spherical_modes(order))} spherical modes; M_(0,0,0) = {multipole[0]:.1e}")
print("first coefficients (alpha, degree, M_alpha):")
for alpha, value in list(zip(indices, multipole))[:6]:
    print(f"  {tuple(int(a) for a in alpha)}  {int(alpha.sum())}  {value: .6e}")

figure, axes = plt.subplots(figsize=(6, 4))
plot_coefficients_by_degree(axes, multipole, order, "P2M coefficient magnitude by degree")
figure.tight_layout()

## M2M: translate a child expansion to its parent

Translating the child multipole to the parent centre reproduces the multipole
built directly at the parent to rounding, because both represent the same
sources exactly up to the common truncation.

In [ ]:
child_centre = np.array([0.25, 0.25, 0.25])
parent_centre = np.zeros(3)
child_positions = child_centre + rng.uniform(-0.2, 0.2, size=(40, 3))
child_moments = rng.normal(size=(40, 3))

child_multipole = cdfmm.p2m_dipole(child_centre, child_positions, child_moments, order=order)
translated = cdfmm.m2m(child_multipole, child_centre, parent_centre, order=order)
direct_parent = cdfmm.p2m_dipole(parent_centre, child_positions, child_moments, order=order)
print("M2M versus direct parent P2M, max coefficient difference:",
      f"{np.max(np.abs(translated - direct_parent)):.2e}")

## M2P: evaluate a multipole at distant targets

M2P is not a stage of the FMM (targets receive the far field through local
expansions), but it is the cleanest way to see how the truncation error
depends on the order and on the ratio of source radius to target distance.

In [ ]:
source_radius = np.max(np.linalg.norm(source_positions, axis=1))
direction = np.array([1.0, 0.35, -0.2]) / np.linalg.norm([1.0, 0.35, -0.2])
distances = np.linspace(1.5, 7.0, 12)
far_targets = distances[:, None] * direction
reference = direct_fields(far_targets, source_positions, dipole_moments)

orders = np.arange(1, 7)
errors = []
for p in orders:
    M = cdfmm.p2m_dipole(source_centre, source_positions, dipole_moments, order=p)
    errors.append(relative_error(multipole_fields(far_targets, M, source_centre, p), reference))
errors = np.asarray(errors)

figure, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].semilogy(orders, np.sqrt(np.mean(errors**2, axis=1)), "o-")
axes[0].set(xlabel="expansion order p", ylabel="RMS relative field error", title="Error versus order")
for index in (0, 2, 5):
    axes[1].semilogy(source_radius / distances, errors[index], "o-", label=f"p = {orders[index]}")
axes[1].set(xlabel="source radius / target distance", ylabel="relative field error",
            title="Error versus geometric ratio")
axes[1].legend()
figure.tight_layout()

## M2L and L2P: local expansions

M2L converts the source multipole into a local Taylor expansion about a
distant target centre; it needs kernel derivatives up to order $2p$. L2P then
evaluates that expansion at targets near the centre. At the centre itself
($dx = 0$) the degree-one local coefficients give $H = -\nabla\phi$ directly.

In [ ]:
target_centre = np.array([4.0, 0.75, -0.5])
local = cdfmm.m2l(multipole, source_centre, target_centre, order=order)
at_centre = cdfmm.l2p(local, target_centre, target_centre, order=order, output="both")
direct_centre = cdfmm.p2p_dipole_sum(target_centre, source_positions, dipole_moments, output="both")
print("L2P at the local centre:", at_centre["H"])
print("direct field:           ", direct_centre["H"])
print(f"relative error {relative_error(at_centre['H'][None], direct_centre['H'][None])[0]:.2e}")

# Spatial error of the local expansion across a plane through its centre,
# compared with M2P from the same multipole.
half_width, points = 0.45, 21
offsets = np.linspace(-half_width, half_width, points)
U, V = np.meshgrid(offsets, offsets)
plane = target_centre + np.stack([U, V, np.zeros_like(U)], axis=-1).reshape(-1, 3)
plane_reference = direct_fields(plane, source_positions, dipole_moments)
l2p_error = relative_error(local_fields(plane, local, target_centre, order), plane_reference)
m2p_error = relative_error(multipole_fields(plane, multipole, source_centre, order), plane_reference)
limits = dict(vmin=np.log10(min(l2p_error.min(), m2p_error.min())),
              vmax=np.log10(max(l2p_error.max(), m2p_error.max())))
figure, axes = plt.subplots(1, 2, figsize=(11, 4.4))
for axis, values, title in ((axes[0], l2p_error, "P2M + M2L + L2P"), (axes[1], m2p_error, "P2M + M2P")):
    image = axis.imshow(np.log10(values).reshape(points, points), origin="lower",
                        extent=[-half_width, half_width, -half_width, half_width], cmap="magma", **limits)
    axis.set(title=title, xlabel="x offset from the local centre", ylabel="y offset")
    figure.colorbar(image, ax=axis, label=r"$\log_{10}$ relative field error")
figure.tight_layout()

The local expansion is most accurate at its centre and degrades towards the
edge of the box; M2P from the source centre has a different, smoother pattern
because it expands about the sources. In the FMM the target box is always well
separated from the sources in its `list2`, which is what keeps the L2P error
bounded.

## L2L and the four complete chains

L2L shifts a parent's local expansion to a child. Independently truncated
M2L expansions at two centres need not have identical coefficients, but the
fields they represent near the child agree. The final plot compares every
route from sources to targets with the direct sum.

In [ ]:
parent_target = np.array([4.0, 0.5, -0.25])
child_target = np.array([4.1, 0.4, -0.15])
child_source_centre = np.array([0.15, -0.10, 0.10])
chain_sources = child_source_centre + rng.uniform(-0.35, 0.35, size=(80, 3))
chain_moments = rng.normal(size=(80, 3))
chain_targets = child_target + rng.uniform(-0.08, 0.08, size=(18, 3))
chain_reference = direct_fields(chain_targets, chain_sources, chain_moments)

routes = {
    "P2M -> M2P": [],
    "P2M -> M2L -> L2P": [],
    "P2M(child) -> M2M -> M2P": [],
    "P2M -> M2L(parent) -> L2L -> L2P": [],
}
chain_orders = np.arange(2, 7)
for p in chain_orders:
    parent_M = cdfmm.p2m_dipole(source_centre, chain_sources, chain_moments, order=p)
    child_M = cdfmm.p2m_dipole(child_source_centre, chain_sources, chain_moments, order=p)
    routes["P2M -> M2P"].append(error_metrics(
        multipole_fields(chain_targets, parent_M, source_centre, p), chain_reference)["rms"])
    child_L = cdfmm.m2l(parent_M, source_centre, child_target, order=p)
    routes["P2M -> M2L -> L2P"].append(error_metrics(
        local_fields(chain_targets, child_L, child_target, p), chain_reference)["rms"])
    translated_M = cdfmm.m2m(child_M, child_source_centre, source_centre, order=p)
    routes["P2M(child) -> M2M -> M2P"].append(error_metrics(
        multipole_fields(chain_targets, translated_M, source_centre, p), chain_reference)["rms"])
    parent_L = cdfmm.m2l(parent_M, source_centre, parent_target, order=p)
    translated_L = cdfmm.l2l(parent_L, parent_target, child_target, order=p)
    routes["P2M -> M2L(parent) -> L2L -> L2P"].append(error_metrics(
        local_fields(chain_targets, translated_L, child_target, p), chain_reference)["rms"])

figure, axes = plt.subplots(figsize=(8, 4.5))
for name, values in routes.items():
    axes.semilogy(chain_orders, values, "o-", label=name)
axes.set(xlabel="expansion order p", ylabel="RMS relative field error",
         title="Every operator route converges to the direct field")
axes.legend(fontsize=8)
figure.tight_layout()

## Inside a static plan

`UniformFmm` runs exactly these operators on the tree. `upward_pass` performs
P2M at the leaves and M2M towards the root; the root multipole equals the P2M
of all sources about the root centre. `downward_pass` performs M2L and L2L;
`evaluate_components` returns the far and near contributions separately and
shows that they add up to the total field.

In [ ]:
options = cdfmm.UniformFmmOptions()
options.expansion_basis = cdfmm.ExpansionBasis.CARTESIAN
options.precision = cdfmm.StaticPrecision.FLOAT64
options.expansion_order = 4
options.tree.max_level = 2
options.tree.root_centre = cdfmm.Vec3(0.0, 0.0, 0.0)
options.tree.root_half_width = 1.0

plan_sources = rng.uniform(-0.95, 0.95, size=(120, 3))
plan_targets = rng.uniform(-0.95, 0.95, size=(40, 3))
plan_moments = rng.normal(size=(120, 3))
with quiet_construction():
    plan = cdfmm.UniformFmm(plan_sources, plan_targets, options)

plan.upward_pass(plan_moments)
root_direct = cdfmm.p2m_dipole(vec3_to_array(plan.tree.root_centre), plan_sources, plan_moments, order=4)
print("root multipole versus direct P2M about the root, relative difference:",
      f"{np.max(np.abs(plan.root_multipole - root_direct)) / np.max(np.abs(root_direct)):.2e}")
occupied = [node for node in plan.tree.nodes if node.is_leaf and node.source_count > 0]
print(f"{len(occupied)} occupied leaves; leaf {occupied[0].index} holds {occupied[0].source_count} sources "
      f"and a multipole of {len(plan.multipole(occupied[0].index))} coefficients")

components = plan.evaluate_components(plan_moments)
reference = direct_fields(plan_targets, plan_sources, plan_moments)
print("far + near == total:", np.allclose(components["H_far"] + components["H_p2p"], components["H_total"]))
print(f"complete FMM versus direct sum, RMS relative error: "
      f"{error_metrics(components['H_total'], reference)['rms']:.2e}")

figure, axes = new_3d_figure(figsize=(7, 6))
leaf = occupied[0]
for node in plan.tree.nodes:
    if node.is_leaf:
        draw_box_3d(axes, vec3_to_array(node.centre), node.half_width, colour="0.85", linewidth=0.4, alpha=0.3)
for slot, index in enumerate(leaf.list2):
    node = plan.tree.nodes[index]
    if node.source_count:
        draw_box_3d(axes, vec3_to_array(node.centre), node.half_width, colour="tab:orange",
                    linewidth=1.0, alpha=0.8, label="list2 sources: M2L" if slot == 0 else None)
for slot, index in enumerate(leaf.list1):
    node = plan.tree.nodes[index]
    draw_box_3d(axes, vec3_to_array(node.centre), node.half_width, colour="tab:blue",
                linewidth=1.3, alpha=0.9, label="list1: exact P2P" if slot == 0 else None)
draw_box_3d(axes, vec3_to_array(leaf.centre), leaf.half_width, colour="tab:red", linewidth=2.5,
            label="selected leaf")
axes.scatter(*plan_sources.T, s=6, color="black", alpha=0.4)
finish_3d_axes(axes, "Near and far partners of one leaf in the static plan")
axes.legend(fontsize=8, loc="upper left")
figure.tight_layout()

## Summary

- Each operator is exact linear algebra on Taylor (or spherical) coefficients;
  only the truncation at order $p$ introduces error, and every route converges
  to the direct field.
- The static plan stores these operators once per geometry and applies them to
  every new moment state; the near field is exact and the far field is the
  M2L/L2L/L2P chain.
- The normative definitions, signs and normalisations are in the mathematics
  pages of the documentation.